In [1]:
%load_ext autoreload
%autoreload 2

# Summary

Collect logprobs for joke dataset. Would have been nice to do this upfront but used gpt-5-mini. Regardless, it would be nice to have this functionality in general.

In [2]:
import gc
import os
from pathlib import Path
from typing import Optional, Union, Any

In [3]:
repo_parent = Path(".").absolute().parent.parent
os.environ["HF_HOME"] = str(repo_parent/".cache")

In [52]:
# Think we need to import after setting env var if we want custom cache dir.
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

from tqdm.auto import tqdm
from datasets import Dataset, load_dataset
import pandas as pd
import numpy as np
from huggingface_hub import login, HfApi

from aeon.secrets import SecretManager
from aeon import config

In [5]:
name = "Qwen/Qwen3-8B-Base"
tokenizer = AutoTokenizer.from_pretrained(name)
model = AutoModelForCausalLM.from_pretrained(name).to("cuda")

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

In [89]:
def get_text_logprobs(prompt: Optional[str], text: str, model, tokenizer, k: int = 10, i2w: Optional[dict] = None):
    """For an existing text sequence (can be LLM-generated, human-written, whatever),
    get some model's logprobs for each token. Essentially shows us how surprising each
    token was.
    
    Parameters
    ----------
    k : int
        Number of most probably tokens to return logprobs for at each step.
    i2w : dict or NoneType
        Maps tokenizer token index (int) to token (str). If not provided, we will
        construct it from the `tokenizer` arg. (Just saves a little time to not have
        to iterate over the whole vocab an extra time on every batch since we want to
        run this func on any inputs.)
    """
    model.generation_config.pad_token_id = tokenizer.pad_token_id
    vocab = tokenizer.get_vocab()
    i2w = i2w or {i: word for word, i in tokenizer.get_vocab().items()}
    tokens = tokenizer.tokenize(text)
    # We will use these as labels later.
    token_idx = torch.tensor([vocab[t] for t in tokens], device=model.device)
    # list[str]
    sequences = [
        tokenizer.convert_tokens_to_string(tokens[:i])
        for i in np.arange(len(tokens))
    ]

    all_inputs = []
    base_messages = [{"role": "system", "content": prompt}] if prompt else []
    for seq in sequences:
        messages = base_messages + [{"role": "user", "content": seq}]
        inputs = tokenizer.apply_chat_template(
        	messages,
            continue_final_message=True,
        	add_generation_prompt=False,
        	tokenize=True,
        	return_dict=True,
        	return_tensors="pt",
        )
        inputs["input_ids"] = inputs["input_ids"].squeeze()
        inputs["attention_mask"] = inputs["attention_mask"].squeeze()
        all_inputs.append(inputs)

    padded_inputs = tokenizer.pad(all_inputs, padding=True, padding_side="left").to(model.device)
    
    outputs = model.generate(**padded_inputs, max_new_tokens=1,
                             return_dict_in_generate=True, output_scores=True)

    # outputs.scores has len max_new_tokens which is always 1 in our case.
    # Just pull out the relevant bit for easy handling.
    # shape: (bs, vocab_size)
    scores = outputs.scores[0]
    logprobs_allrows = scores.log_softmax(dim=-1)
    # logprob for correct next token for each row.
    label_logprobs = logprobs_allrows[torch.arange(logprobs_allrows.shape[0]).to(model.device), token_idx]
    
    # Get index of top 10 logprobs for each row
    idx_allrows = logprobs_allrows.argsort(dim=-1, descending=True)
    label_rank = (idx_allrows == token_idx.unsqueeze(-1)).nonzero()[:, -1]
    idx_topk = idx_allrows[:, :k]
    logprobs_topk = logprobs_allrows.gather(-1, idx_topk)

    res = []
    for label, label_logprob, rank, idx, logprobs in zip(
        tokens, label_logprobs, label_rank, idx_topk, logprobs_topk
    ):
        probs = logprobs.exp()
        item = {
            "label": label,
            "label_prob": label_logprob.exp().item(),
            "label_rank": rank.item(),
            "label_logprob": label_logprob.item(),
            "top_k_probs": {
                i2w[i.item()]: prob.item() for i, prob in zip(idx, probs)
            },
            "top_k_logprobs": {
                i2w[i.item()]: logprob.item() for i, logprob in zip(idx, logprobs)
            },
        }
        res.append(item)
    return res

In [86]:
vocab = tokenizer.get_vocab()
i2w = {i: word for word, i in vocab.items()}

In [87]:
res = get_text_logprobs("Tell me about the sky.", "The sky is blue and grass is green.", model, tokenizer, i2w=i2w)

In [28]:
pd.DataFrame(res)

,label,label_prob,label_rank,label_logprob,top_k_probs,top_k_logprobs
0,The,0.171236,0,-1.764715,"{'The': 0.1712355613708496, 'Tell': 0.14073315...","{'The': -1.7647150754928589, 'Tell': -1.960889..."
1,Ġsky,0.933374,0,-0.068950,"{'Ġsky': 0.9333735108375549, 'ĠSky': 0.0103762...","{'Ġsky': -0.06894978135824203, 'ĠSky': -4.5682..."
2,Ġis,0.705182,0,-0.349300,"{'Ġis': 0.7051815986633301, ',': 0.10386520624...","{'Ġis': -0.34929996728897095, ',': -2.26466131..."
3,Ġblue,0.046402,2,-3.070406,"{'Ġa': 0.4470357596874237, 'Ġthe': 0.230530753...","{'Ġa': -0.8051167130470276, 'Ġthe': -1.4673709..."
4,Ġand,0.097452,2,-2.328398,"{'.': 0.23665662109851837, '.Ċ': 0.19150346517...","{'.': -1.4411450624465942, '.Ċ': -1.6528493165..."
5,Ġgrass,0.000010,821,-11.465742,"{'Ġwhite': 0.13064095377922058, 'Ġthe': 0.1048...","{'Ġwhite': -2.0353026390075684, 'Ġthe': -2.255..."
6,Ġis,0.887614,0,-0.119219,"{'Ġis': 0.887613832950592, 'y': 0.066946402192...","{'Ġis': -0.11921855807304382, 'y': -2.70386290..."
7,Ġgreen,0.951815,0,-0.049384,"{'Ġgreen': 0.9518154859542847, 'Ġblue': 0.0056...","{'Ġgreen': -0.04938405752182007, 'Ġblue': -5.1..."
8,.,0.291438,1,-1.232930,"{'.Ċ': 0.32165002822875977, '.': 0.29143750667...","{'.Ċ': -1.134291172027588, '.': -1.23292970657..."


In [16]:
ds = load_dataset("hmamin/extract_jokes")

In [17]:
df = ds['train'].to_pandas()

In [18]:
df.tail()

,prompt,joke,subtext,unfunny_variant,web_scraper_order,transcript_link,transcript_link_href
22924,Any better short-term investment ideas?,"Why don’t you name a better way to make $6,000...",Foolish financial decisions can be rationalize...,I rationalized a bad financial loss as a way t...,1686242983-419,John Mulaney: Baby J (2023) | Transcript,https://scrapsfromtheloft.com/comedy/john-mula...
22925,How has recovery changed your self-image?,It’s weird to be a recovering drug addict... S...,Surviving personal harm can shift priorities a...,"Having survived addiction, I'm less concerned ...",1686242983-419,John Mulaney: Baby J (2023) | Transcript,https://scrapsfromtheloft.com/comedy/john-mula...
22926,Any awkward public parenting moments?,I was in a museum in Detroit with my son and n...,Parents can be confronted with reminders of pa...,"While changing my baby in a museum restroom, I...",1686242983-419,John Mulaney: Baby J (2023) | Transcript,https://scrapsfromtheloft.com/comedy/john-mula...
22927,Do you remember things you said on drugs?,"I gave an interview to GQ December 15th, 2020 ...",Substance use impairs memory and leads to inco...,I don't remember certain interviews I apparent...,1686242983-419,John Mulaney: Baby J (2023) | Transcript,https://scrapsfromtheloft.com/comedy/john-mula...
22928,"If you had a talk show, what would it be like?","GQ asked if I'd want my own talk show. I said,...",Some talk show concepts are oddly specific or ...,"I once thought about two talk show concepts, i...",1686242983-419,John Mulaney: Baby J (2023) | Transcript,https://scrapsfromtheloft.com/comedy/john-mula...


# Idea:

maybe should allow excluding the first n tokens from the logprob exercise. Like I could pass in f"{prompt}\n{joke}" and keep {prompt} fixed, so the first token of joke is conditioned on prompt rather than on nothing.

In [34]:
row

,prompt,joke,subtext,unfunny_variant,web_scraper_order,transcript_link,transcript_link_href
21545,Have ghost hunter shows actually found evidence?,You’ve never seen a fucking ghost. Not one. It...,Decades of ghost-hunting TV have produced no d...,"Ghost shows often produce inconclusive, silly ...",1686242875-393,Ricky Gervais: SuperNature (2022) | Transcript,https://scrapsfromtheloft.com/comedy/ricky-ger...


In [31]:
row = df.sample(1)

In [32]:
res = get_text_logprobs(row.prompt.values[0], row.joke.values[0], model, tokenizer, i2w=i2w)

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


In [33]:
pd.DataFrame(res)

,label,label_prob,label_rank,label_logprob,top_k_probs,top_k_logprobs
0,You,0.007833,17,-4.849458,"{'Yes': 0.08825743198394775, 'I': 0.0826682671...","{'Yes': -2.427497386932373, 'I': -2.4929194450..."
1,âĢĻve,0.003204,38,-5.743373,"{''re': 0.1151023581624031, 'Ġare': 0.11087415...","{''re': -2.161933422088623, 'Ġare': -2.1993594..."
2,Ġnever,0.020481,6,-3.888256,"{'Ġgot': 0.1725987195968628, 'Ġprobably': 0.13...","{'Ġgot': -1.7567859888076782, 'Ġprobably': -1...."
3,Ġseen,0.332288,0,-1.101754,"{'Ġseen': 0.33228763937950134, 'Ġbeen': 0.1000...","{'Ġseen': -1.1017543077468872, 'Ġbeen': -2.301..."
4,Ġa,0.330201,0,-1.108055,"{'Ġa': 0.3302007019519806, 'Ġanything': 0.0814...","{'Ġa': -1.1080546379089355, 'Ġanything': -2.50..."
5,Ġfucking,0.000131,161,-8.937785,"{'Ġghost': 0.8041595816612244, 'Ġreal': 0.0216...","{'Ġghost': -0.21795758605003357, 'Ġreal': -3.8..."
6,Ġghost,0.873207,0,-0.135582,"{'Ġghost': 0.8732073903083801, 'Ġthing': 0.011...","{'Ġghost': -0.13558222353458405, 'Ġthing': -4...."
7,.,0.188246,0,-1.670005,"{'.': 0.18824611604213715, ',': 0.164621725678...","{'.': -1.6700050830841064, ',': -1.80410504341..."
8,ĠNot,0.008287,21,-4.793097,"{'ĠYou': 0.117436483502388, 'ĠI': 0.0466322675...","{'ĠYou': -2.141857624053955, 'ĠI': -3.06546258..."
9,Ġone,0.181056,1,-1.708951,"{'Ġeven': 0.22972415387630463, 'Ġone': 0.18105...","{'Ġeven': -1.4708760976791382, 'Ġone': -1.7089..."


In [65]:
out_dir = config.DATA_DIR/'tmp'

In [71]:
model.config.pad_token_id

In [72]:
tokenizer.pad_token_id

151643

In [73]:
tokenizer.eos_token_id

151643

In [ ]:
all_res = []
for i, row in tqdm(df.iterrows()):
    try:
        res = get_text_logprobs(row.prompt, row.joke, model, tokenizer, i2w=i2w)
        df_res = pd.DataFrame(res)
        df_res.to_parquet(out_dir/f"{i}.pq")
    except Exception as e:
        print(f'[row {i}] error: {e}')
    else:
        all_res.append(df_res)

    # Progress bar rendering is flaky, add a backup way to monitor progress.
    # In practice `watch`ing the data/tmp dir is probably better though.
    if not i % 100:
        print(i)

0it [00:00, ?it/s]

0
100
200
300
400
500
600
700
800
900
1000
1100
1200


In [ ]:
df_all = pd.concat([dfi.assign(id=i) for i, dfi in enumerate(all_res)], axis=0).reset_index(drop=True)